In [ ]:
from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Iterable

import numpy as np


# 仓库根目录：默认取 notebook 的当前工作目录（请在仓库根目录启动 Jupyter），
# 也可用环境变量 V2VA_REPO_ROOT / V2VA_DATA_ROOT 覆盖。
REPO_ROOT = Path(os.environ.get("V2VA_REPO_ROOT", Path.cwd())).resolve()
DEFAULT_DATA_DIR = Path(os.environ.get("V2VA_DATA_ROOT", REPO_ROOT / "data"))


def _pick_image_key(npz: np.lib.npyio.NpzFile) -> str | None:
	for key in ("train_images", "val_images", "test_images", "images"):
		if key in npz:
			return key
	return None


def _pick_label_key(npz: np.lib.npyio.NpzFile, image_key: str) -> str | None:
	mapping = {
		"train_images": "train_labels",
		"val_images": "val_labels",
		"test_images": "test_labels",
		"images": "labels",
	}
	prefer = mapping.get(image_key)
	if prefer and prefer in npz:
		return prefer
	for key in ("train_labels", "val_labels", "test_labels", "labels"):
		if key in npz:
			return key
	return None


def _to_volume(sample: np.ndarray) -> np.ndarray:
	"""Convert sample to a 3D volume (D, H, W order unknown but 3D guaranteed)."""
	if sample.ndim == 3:
		return sample

	if sample.ndim == 4:
		# Common channel-last or channel-first with singleton channel.
		if sample.shape[-1] == 1:
			return sample[..., 0]
		if sample.shape[0] == 1:
			return sample[0]

		raise ValueError(
			f"4D sample has non-singleton channel dimension, unsupported shape={sample.shape}"
		)

	raise ValueError(f"Sample should be 3D/4D, got shape={sample.shape}")


def _parse_slice_indices(text: str | None) -> list[int] | None:
	if not text:
		return None
	indices = []
	for x in text.split(","):
		x = x.strip()
		if not x:
			continue
		indices.append(int(x))
	return indices or None


def _auto_indices(length: int, num_slices: int) -> list[int]:
	num_slices = max(1, min(num_slices, length))
	idx = np.linspace(0, length - 1, num_slices, dtype=int)
	return sorted(set(int(i) for i in idx))


def _normalize_indices(length: int, indices: Iterable[int]) -> list[int]:
	out = []
	for i in indices:
		if i < 0:
			i = length + i
		if i < 0 or i >= length:
			raise IndexError(f"slice index {i} out of range [0, {length - 1}]")
		out.append(int(i))
	return sorted(set(out))


def _print_slice(slice_2d: np.ndarray, index: int, show_values: bool) -> None:
	print(
		f"    slice[{index:>3}] shape={slice_2d.shape} "
		f"min={float(slice_2d.min()):.4f} "
		f"max={float(slice_2d.max()):.4f} "
		f"mean={float(slice_2d.mean()):.4f}"
	)
	if show_values:
		print(np.array2string(slice_2d, precision=3, suppress_small=True, max_line_width=180))


def inspect_one_dataset(
	file_path: Path,
	axis: int,
	sample_index: int,
	num_slices: int,
	slice_indices: list[int] | None,
	show_values: bool,
) -> None:
	with np.load(file_path, allow_pickle=False) as npz:
		image_key = _pick_image_key(npz)
		if image_key is None:
			print(f"[SKIP] {file_path.name}: no image keys found")
			return

		images = npz[image_key]
		if images.ndim < 4:
			print(f"[SKIP] {file_path.name}: image array ndim={images.ndim} < 4")
			return

		if sample_index < 0:
			index = images.shape[0] + sample_index
		else:
			index = sample_index
		if index < 0 or index >= images.shape[0]:
			print(
				f"[SKIP] {file_path.name}: sample index {sample_index} out of range "
				f"for N={images.shape[0]}"
			)
			return

		sample = images[index]
		volume = _to_volume(sample)

		if axis < -volume.ndim or axis >= volume.ndim:
			raise ValueError(f"axis={axis} out of range for volume ndim={volume.ndim}")
		axis_norm = axis % volume.ndim
		depth = volume.shape[axis_norm]

		if slice_indices is None:
			indices = _auto_indices(depth, num_slices)
		else:
			indices = _normalize_indices(depth, slice_indices)

		label_key = _pick_label_key(npz, image_key)
		label_text = "<none>"
		if label_key is not None:
			labels = npz[label_key]
			if labels.shape[0] > index:
				label_text = str(labels[index].reshape(-1).tolist())

		print("=" * 88)
		print(f"Dataset  : {file_path.stem}")
		print(f"Image key: {image_key}, Label key: {label_key}")
		print(f"Images   : shape={images.shape}, dtype={images.dtype}")
		print(f"Sample   : index={index}, sample_shape={sample.shape}, volume_shape={volume.shape}")
		print(f"Label    : {label_text}")
		print(f"Slice cfg: axis={axis_norm}, depth={depth}, indices={indices}")

		for i in indices:
			sl = np.take(volume, i, axis=axis_norm)
			_print_slice(sl, i, show_values=show_values)


In [ ]:
def main() -> None:
	parser = argparse.ArgumentParser(
		description="每个 3D 子数据集选取一个样本，并按指定切片维度打印切片信息。"
	)
	parser.add_argument("--data-dir", type=Path, default=DEFAULT_DATA_DIR, help="数据目录，默认 data/")
	parser.add_argument("--axis", type=int, default=0, help="切片维度（可用负数），默认 0")
	parser.add_argument("--sample-index", type=int, default=0, help="每个子数据集选取的样本索引，默认 0")
	parser.add_argument("--num-slices", type=int, default=8, help="自动均匀选取的切片数量，默认 8")
	parser.add_argument(
		"--slice-indices",
		type=str,
		default=None,
		help="手动指定切片索引，逗号分隔，如 0,10,20（优先于 --num-slices）",
	)
	parser.add_argument(
		"--show-values",
		action="store_true",
		help="打印每个切片的二维像素矩阵（默认仅打印统计信息）",
	)
	args = parser.parse_args()

	if not args.data_dir.exists():
		raise FileNotFoundError(f"data dir not found: {args.data_dir}")

	files = sorted(args.data_dir.glob("*.npz"))
	if not files:
		print(f"No npz files found in {args.data_dir}")
		return

	manual_indices = _parse_slice_indices(args.slice_indices)

	print(f"Found {len(files)} datasets in {args.data_dir}")
	for fp in files:
		inspect_one_dataset(
			file_path=fp,
			axis=args.axis,
			sample_index=args.sample_index,
			num_slices=args.num_slices,
			slice_indices=manual_indices,
			show_values=args.show_values,
		)


if __name__ == "__main__":
	main()


In [ ]:
# 导出：每个数据集一个样本（保存为 .npz）
from pathlib import Path

import numpy as np

data_dir = DEFAULT_DATA_DIR
out_dir = Path(os.environ.get("V2VA_EXPORT_SAMPLES_DIR", REPO_ROOT / "outputs" / "export_samples"))
sample_index = 0  # 可改为其他索引，支持负数

out_dir.mkdir(parents=True, exist_ok=True)
files = sorted(data_dir.glob("*.npz"))

if not files:
    print(f"No npz files found in {data_dir}")
else:
    print(f"Found {len(files)} datasets")
    for fp in files:
        with np.load(fp, allow_pickle=False) as npz:
            image_key = _pick_image_key(npz)
            if image_key is None:
                print(f"[SKIP] {fp.name}: no image key")
                continue

            images = npz[image_key]
            n = images.shape[0]
            idx = sample_index if sample_index >= 0 else n + sample_index
            if idx < 0 or idx >= n:
                print(f"[SKIP] {fp.name}: sample_index={sample_index} out of range, N={n}")
                continue

            x = images[idx]
            label_key = _pick_label_key(npz, image_key)
            y = None
            if label_key is not None and npz[label_key].shape[0] > idx:
                y = npz[label_key][idx]

            save_path = out_dir / f"{fp.stem}_sample{idx}.npz"
            if y is None:
                np.savez_compressed(save_path, image=x)
                print(f"[OK] {fp.stem}: saved image only -> {save_path}")
            else:
                np.savez_compressed(save_path, image=x, label=y)
                print(f"[OK] {fp.stem}: saved image+label -> {save_path}")

    print(f"Done. Output dir: {out_dir}")


In [ ]:
# 导出给 3D Slicer：每个数据集一个样本，保存为 .nii.gz
# 说明：Slicer 不能直接加载 .npz，需要 NIfTI/NRRD 等医学影像格式
from pathlib import Path

import numpy as np

try:
    import nibabel as nib
except Exception as e:
    raise ImportError(
        "缺少 nibabel，请先安装：pip install nibabel\n"
        f"原始错误: {e}"
    )

data_dir = DEFAULT_DATA_DIR
out_dir = Path(os.environ.get("V2VA_EXPORT_NII_DIR", REPO_ROOT / "outputs" / "export_slicer_nii"))
sample_index = 1  # 每个数据集导出的样本索引
transpose_to_xyz = True  # True: (D,H,W)->(W,H,D)，更符合 Slicer 常见显示方向

out_dir.mkdir(parents=True, exist_ok=True)
files = sorted(data_dir.glob("*.npz"))

if not files:
    print(f"No npz files found in {data_dir}")
else:
    print(f"Found {len(files)} datasets")
    for fp in files:
        with np.load(fp, allow_pickle=False) as npz:
            image_key = _pick_image_key(npz)
            if image_key is None:
                print(f"[SKIP] {fp.name}: no image key")
                continue

            images = npz[image_key]
            n = images.shape[0]
            idx = sample_index if sample_index >= 0 else n + sample_index
            if idx < 0 or idx >= n:
                print(f"[SKIP] {fp.name}: sample_index={sample_index} out of range, N={n}")
                continue

            x = _to_volume(images[idx]).astype(np.float32)  # 3D
            if transpose_to_xyz:
                x = np.transpose(x, (2, 1, 0))

            img_nii = nib.Nifti1Image(x, affine=np.eye(4, dtype=np.float32))
            img_path = out_dir / f"{fp.stem}_sample{idx}_img.nii.gz"
            nib.save(img_nii, str(img_path))

            label_key = _pick_label_key(npz, image_key)
            if label_key is not None and npz[label_key].shape[0] > idx:
                y = np.asarray(npz[label_key][idx]).reshape(-1)
                cls = int(y[0]) if y.size > 0 else -1
                txt_path = out_dir / f"{fp.stem}_sample{idx}_label.txt"
                txt_path.write_text(str(cls), encoding="utf-8")
                print(f"[OK] {fp.stem}: {img_path.name} + {txt_path.name} (class={cls})")
            else:
                print(f"[OK] {fp.stem}: {img_path.name} (no label)")

    print(f"Done. Output dir: {out_dir}")
